# Seed07 Unlabeled Pseudo-Labeling

This notebook uses the saved `seed07` multiclass classifier to label the **unlabeled** rows in WM-811K.

Goal:
- run the `seed07` classifier on unlabeled WM-811K rows
- attach a pseudo label plus confidence for each row
- save a reusable pseudo-label table for later validation work
- export two classifier-space UMAP views:
  - the standard pseudo-label projection
  - a `10A`-style PCA -> UMAP view using the same UMAP recipe as the anomaly notebook

Kaggle inputs needed:
- this classifier bundle dataset
- the raw WM-811K dataset that contains `LSWMD.pkl`


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import shutil
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display


BUNDLE_SOURCE = Path("/kaggle/input/seed07_pseudolabel_bundle_kaggle/seed07_pseudolabel_bundle_kaggle")  # INSERT BUNDLE PATH HERE
RAW_PICKLE = Path("/kaggle/input/wm811k/LSWMD.pkl")  # INSERT RAW PICKLE PATH HERE
REPO_ROOT = Path("/kaggle/working/seed07_pseudolabel_bundle_kaggle")


def run_logged(title: str, command: list[str], env: dict[str, str], cwd: Path) -> None:
    print(f"\n{'=' * 80}\n{title}\n{'=' * 80}")
    print("Command:", " ".join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)
    print(f"Completed: {title}\n")


if not BUNDLE_SOURCE.exists():
    raise FileNotFoundError(
        f"Bundle source path does not exist: {BUNDLE_SOURCE}\n"
        "Edit BUNDLE_SOURCE in this cell to the mounted Kaggle bundle path."
    )

if not RAW_PICKLE.exists():
    raise FileNotFoundError(
        f"Raw pickle path does not exist: {RAW_PICKLE}\n"
        "Edit RAW_PICKLE in this cell to the mounted WM-811K pickle path."
    )

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
shutil.copytree(BUNDLE_SOURCE, REPO_ROOT)
os.chdir(REPO_ROOT)

if not (REPO_ROOT / "src" / "wafer_defect").exists():
    raise RuntimeError(f"Copied bundle does not look like a repo root: {REPO_ROOT}")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("Repo root:", REPO_ROOT)
print("Bundle source:", BUNDLE_SOURCE)
print("Raw pickle:", RAW_PICKLE)
print("Working directory:", Path.cwd().resolve())


In [ ]:
base_data_config_path = REPO_ROOT / "configs/data/classifier/data_multiclass_all_80_10_10.toml"

runtime_dir = REPO_ROOT / "runtime_configs"
runtime_dir.mkdir(parents=True, exist_ok=True)
data_config_runtime = runtime_dir / "data_multiclass_all_80_10_10_pseudolabel.kaggle.toml"

config_text = base_data_config_path.read_text(encoding="utf-8")
config_text = config_text.replace(
    'raw_pickle = "data/raw/LSWMD.pkl"',
    f'raw_pickle = "{RAW_PICKLE.as_posix()}"',
)
data_config_runtime.write_text(config_text, encoding="utf-8")

checkpoint_path = REPO_ROOT / "artifacts/multiclass_classifier_all_80_10_10_seed07/best_model.pt"
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Missing seed07 checkpoint inside bundle: {checkpoint_path}")

output_dir = REPO_ROOT / "artifacts/multiclass_classifier_all_80_10_10_seed07_pseudolabels"
raw_predictions_csv = output_dir / "unlabeled_predictions.seed07.raw.csv"
enriched_predictions_csv = output_dir / "unlabeled_predictions.seed07.pseudolabels.csv"
accepted_predictions_csv = output_dir / "unlabeled_predictions.seed07.accepted.csv"
summary_json = output_dir / "unlabeled_predictions.seed07.summary.json"
standard_umap_dir = output_dir / "umap_visualization"
umap_10a_dir = output_dir / "umap_10a_style"

confidence_threshold = 0.98
limit = None  # Set an integer if you want a smaller smoke run.

output_dir.mkdir(parents=True, exist_ok=True)

print("Base data config:", base_data_config_path)
print("Runtime data config:", data_config_runtime)
print("Raw pickle:", RAW_PICKLE)
print("Checkpoint:", checkpoint_path)
print("Output dir:", output_dir)
print("Confidence threshold:", confidence_threshold)
print("Limit:", limit)


In [ ]:
predict_command = [
    sys.executable,
    "-u",
    str(REPO_ROOT / "scripts/classifier/predict_unlabeled_multiclass.py"),
    "--config",
    str(data_config_runtime),
    "--checkpoint",
    str(checkpoint_path),
    "--output-csv",
    str(raw_predictions_csv),
    "--min-confidence",
    str(confidence_threshold),
]
if limit is not None:
    predict_command.extend(["--limit", str(limit)])

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT / "src")
env["PYTHONUNBUFFERED"] = "1"

run_logged("Seed07 pseudo-label unlabeled WM-811K rows", predict_command, env=env, cwd=REPO_ROOT)


In [ ]:
predictions = pd.read_csv(raw_predictions_csv)

predictions = predictions.rename(
    columns={
        "predicted_label": "pseudo_label",
        "predicted_index": "pseudo_label_index",
        "confidence": "pseudo_label_confidence",
    }
)
predictions["pseudo_label_confidence_pct"] = predictions["pseudo_label_confidence"] * 100.0
predictions["second_choice_confidence_pct"] = predictions["second_choice_confidence"] * 100.0
predictions["confidence_margin"] = (
    predictions["pseudo_label_confidence"] - predictions["second_choice_confidence"]
)
predictions["predicted_is_defect"] = predictions["pseudo_label"] != "none"
predictions["checkpoint_name"] = checkpoint_path.parent.name

accepted_predictions = predictions[predictions["accepted_for_pseudo_label"]].copy()
accepted_predictions = accepted_predictions.sort_values(
    ["pseudo_label_confidence", "confidence_margin"],
    ascending=[False, False],
)

predictions.to_csv(enriched_predictions_csv, index=False)
accepted_predictions.to_csv(accepted_predictions_csv, index=False)

summary = {
    "checkpoint": str(checkpoint_path),
    "data_config_runtime": str(data_config_runtime),
    "source_predictions_csv": str(raw_predictions_csv),
    "enriched_predictions_csv": str(enriched_predictions_csv),
    "accepted_predictions_csv": str(accepted_predictions_csv),
    "confidence_threshold": float(confidence_threshold),
    "rows_scored": int(len(predictions)),
    "accepted_rows": int(len(accepted_predictions)),
    "accepted_fraction": float(accepted_predictions.shape[0] / max(1, predictions.shape[0])),
    "predicted_label_distribution": predictions["pseudo_label"].value_counts().sort_index().to_dict(),
    "accepted_label_distribution": accepted_predictions["pseudo_label"].value_counts().sort_index().to_dict(),
    "predicted_defect_fraction": float(predictions["predicted_is_defect"].mean()),
    "accepted_defect_fraction": float(accepted_predictions["predicted_is_defect"].mean()) if len(accepted_predictions) else 0.0,
    "mean_confidence": float(predictions["pseudo_label_confidence"].mean()),
    "mean_accepted_confidence": float(accepted_predictions["pseudo_label_confidence"].mean()) if len(accepted_predictions) else 0.0,
}
summary_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Saved enriched pseudo labels to:", enriched_predictions_csv)
print("Saved accepted pseudo labels to:", accepted_predictions_csv)
print("Saved pseudo-label summary to:", summary_json)


In [ ]:
display(predictions.head())
display(predictions[["pseudo_label_confidence", "pseudo_label_confidence_pct", "confidence_margin"]].describe())
display(predictions["pseudo_label"].value_counts().rename_axis("pseudo_label").to_frame("count"))
display(accepted_predictions["pseudo_label"].value_counts().rename_axis("pseudo_label").to_frame("accepted_count"))
print("Accepted fraction:", summary["accepted_fraction"])
print("Predicted defect fraction:", summary["predicted_defect_fraction"])
print("Accepted defect fraction:", summary["accepted_defect_fraction"])


In [ ]:
review_columns = [
    "raw_index",
    "pseudo_label",
    "pseudo_label_confidence",
    "pseudo_label_confidence_pct",
    "second_choice_label",
    "second_choice_confidence",
    "second_choice_confidence_pct",
    "confidence_margin",
    "accepted_for_pseudo_label",
    "predicted_is_defect",
]

print("Most confident pseudo labels")
display(predictions.sort_values("pseudo_label_confidence", ascending=False)[review_columns].head(20))

print("Least confident pseudo labels")
display(predictions.sort_values("pseudo_label_confidence", ascending=True)[review_columns].head(20))

thresholds_to_review = [0.50, 0.75, confidence_threshold]
threshold_summary_rows = []

for threshold in thresholds_to_review:
    subset = predictions[predictions["pseudo_label_confidence"] >= threshold].copy()
    threshold_summary_rows.append(
        {
            "confidence_threshold": threshold,
            "confidence_threshold_pct": threshold * 100.0,
            "accepted_rows": int(len(subset)),
            "accepted_fraction": float(len(subset) / max(1, len(predictions))),
            "accepted_defect_rows": int(subset["predicted_is_defect"].sum()),
            "accepted_none_rows": int((~subset["predicted_is_defect"]).sum()),
        }
    )

threshold_summary = pd.DataFrame(threshold_summary_rows)
display(threshold_summary)

for threshold in thresholds_to_review:
    subset = predictions[predictions["pseudo_label_confidence"] >= threshold].copy()
    print(f"Pseudo labels with confidence >= {threshold:.0%}: {len(subset)} ({len(subset) / max(1, len(predictions)):.2%})")
    display(subset["pseudo_label"].value_counts().rename_axis("pseudo_label").to_frame("count"))


In [ ]:
if importlib.util.find_spec("umap") is None:
    install_command = [sys.executable, "-m", "pip", "install", "umap-learn"]
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    run_logged("Install umap-learn", install_command, env=env, cwd=REPO_ROOT)
else:
    print("umap-learn is already available.")


In [ ]:
standard_umap_command = [
    sys.executable,
    "-u",
    str(REPO_ROOT / "scripts/classifier/export_seed07_pseudolabel_umap.py"),
    "--config",
    str(data_config_runtime),
    "--checkpoint",
    str(checkpoint_path),
    "--pseudo-label-csv",
    str(enriched_predictions_csv),
    "--output-dir",
    str(standard_umap_dir),
    "--batch-size",
    "256",
    "--max-labeled-per-class",
    "400",
    "--max-pseudo-per-class",
    "800",
    "--min-confidence",
    "0.75",
    "--n-neighbors",
    "30",
    "--min-dist",
    "0.1",
    "--metric",
    "cosine",
    "--random-seed",
    "7",
]

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT / "src")
env["PYTHONUNBUFFERED"] = "1"

run_logged("Export standard classifier UMAP", standard_umap_command, env=env, cwd=REPO_ROOT)

standard_summary_path = standard_umap_dir / "seed07_pseudolabel_umap.summary.json"
standard_summary = json.loads(standard_summary_path.read_text(encoding="utf-8"))
display(pd.DataFrame([standard_summary]))
display(Image(filename=str(standard_umap_dir / "seed07_pseudolabel_umap.png")))


In [ ]:
umap_10a_command = [
    sys.executable,
    "-u",
    str(REPO_ROOT / "scripts/classifier/export_seed07_pseudolabel_umap_10a_style.py"),
    "--config",
    str(data_config_runtime),
    "--checkpoint",
    str(checkpoint_path),
    "--pseudo-label-csv",
    str(enriched_predictions_csv),
    "--output-dir",
    str(umap_10a_dir),
    "--batch-size",
    "256",
    "--max-labeled-normal",
    "4000",
    "--max-labeled-defect",
    "4000",
    "--max-pseudo",
    "8000",
    "--min-confidence",
    "0.75",
    "--n-neighbors",
    "15",
    "--min-dist",
    "0.10",
    "--metric",
    "euclidean",
    "--random-seed",
    "42",
    "--max-pca-dim",
    "50",
]

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT / "src")
env["PYTHONUNBUFFERED"] = "1"

run_logged("Export 10A-style classifier UMAP", umap_10a_command, env=env, cwd=REPO_ROOT)

umap_10a_summary_path = umap_10a_dir / "umap_10a_style.summary.json"
umap_10a_summary = json.loads(umap_10a_summary_path.read_text(encoding="utf-8"))
display(pd.DataFrame([umap_10a_summary]))
display(Image(filename=str(umap_10a_dir / "umap_by_split_10a_style.png")))
display(Image(filename=str(umap_10a_dir / "umap_by_score_10a_style.png")))
display(Image(filename=str(umap_10a_dir / "umap_by_pseudo_label_10a_style.png")))
